In [1]:
import pandas as pd
import numpy as np
import holidays
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

PORTFOLIOS  = ['A', 'B', 'C', 'D']
DATA_DIR    = './data'
OUTPUT_PATH = './forecast_lstm_v03.csv'

# Targets match original LSTM scope (4 outputs)
TARGETS = ['Call Volume', 'CCT', 'Service Level', 'Abandon Rate']
NUM_TARGETS = len(TARGETS)

# Feature set mirrors XGBoost DAILY_FEATURES, plus LSTM-specific additions
# (Workload, is_holiday). One-hot DayOfWeek is replaced by cyclical encodings.
FEATURE_COLS = [
    'day_of_week', 'month', 'day_of_month', 'week_of_month', 'is_weekend',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'is_holiday',
    'Workload',                                        # LSTM: CV × CCT signal
    'cv_lag_364', 'cct_lag_364', 'abd_lag_364',        # XGB: same-DOW prior year
    'aug_cv_mean', 'aug_cct_mean', 'aug_abd_mean',     # XGB: August overall anchor
    'aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean',  # XGB: August DOW anchor
]

# Scaler input column order: targets first, then features.
# This order must be identical at fit time, inference time, and inverse_transform time.
ALL_COLS = TARGETS + FEATURE_COLS


# ══════════════════════════════════════════════════════════════════════════════
# ASYMMETRIC LOSS  (from original LSTM — kept intact)
# Penalises under-forecasting 3× more than over-forecasting, matching the
# workload-penalty rules where understaffing costs more than overstaffing.
# ══════════════════════════════════════════════════════════════════════════════
def asymmetric_loss(y_true, y_pred):
    error = y_true - y_pred
    loss  = K.maximum(3.0 * error, -1.0 * error)
    return K.mean(loss, axis=-1)


# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN DAILY DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_daily(portfolio):
    df = pd.read_csv(f'{DATA_DIR}/{portfolio} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()

    # Robust date parse: take first 8 chars to strip any trailing characters
    df['Date']         = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df                 = df.sort_values('Date').reset_index(drop=True)

    # ── Calendar features ────────────────────────────────────────────────────
    df['day_of_week']  = df['Date'].dt.dayofweek          # 0=Mon … 6=Sun
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    # ── Cyclical encodings ───────────────────────────────────────────────────
    # Replaces get_dummies(DayOfWeek). Cyclical encoding keeps Sunday (6) and
    # Monday (0) adjacent in feature space, which one-hot encoding cannot do.
    # This is strictly better for sequence models like LSTM.
    df['dow_sin']   = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # ── Holiday flag ─────────────────────────────────────────────────────────
    us_holidays      = holidays.US(years=[2024, 2025, 2026])
    df['is_holiday'] = df['Date'].isin(us_holidays).astype(int)

    # ── Three-level null imputation (from XGBoost model) ────────────────────
    # Level 1: same day-of-week median (preserves weekly seasonality)
    # Level 2: same month median (preserves monthly seasonality)
    # Level 3: global median (last-resort fallback)
    for col in ['Call Volume', 'CCT', 'Service Level', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df.groupby('month')[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)

    # ── Workload feature (from original LSTM) ────────────────────────────────
    # Gives the model a direct signal about staffing load rather than making it
    # multiply CV and CCT implicitly.
    df['Workload'] = df['Call Volume'] * df['CCT']

    # ── 364-day lag features (from XGBoost model) ────────────────────────────
    # 364 = 52 × 7: exactly one year back while preserving day-of-week alignment.
    # For August 2026 inference these will be August 2025 actuals — strong anchors.
    df['cv_lag_364']  = df['Call Volume'].shift(364)
    df['cct_lag_364'] = df['CCT'].shift(364)
    df['abd_lag_364'] = df['Abandon Rate'].shift(364)

    # The first ~364 rows have no lag yet; fill them with DOW median so the
    # LSTM can still train on those rows without NaN inputs.
    for col in ['cv_lag_364', 'cct_lag_364', 'abd_lag_364']:
        df[col] = df.groupby('day_of_week')[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median())

    # ── August baseline features (from XGBoost model) ────────────────────────
    # Anchors predictions in historically observed August behaviour, preventing
    # the model from extrapolating blindly into an unseen forecast month.
    aug_data           = df[df['month'] == 8]
    df['aug_cv_mean']  = aug_data['Call Volume'].mean()
    df['aug_cct_mean'] = aug_data['CCT'].mean()
    df['aug_abd_mean'] = aug_data['Abandon Rate'].mean()

    aug_dow          = aug_data.groupby('day_of_week')[['Call Volume', 'CCT', 'Abandon Rate']].mean()
    aug_dow.columns  = ['aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean']
    df               = df.merge(aug_dow, on='day_of_week', how='left')

    for col in ['aug_cv_mean', 'aug_cct_mean', 'aug_abd_mean',
                'aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean']:
        df[col] = df[col].fillna(df[col].median())

    remaining = df[['Call Volume', 'CCT', 'Service Level', 'Abandon Rate']].isnull().sum().sum()
    print(f"  {portfolio} daily nulls after cleanup: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN INTERVAL DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_interval(portfolio):
    df = pd.read_csv(f'{DATA_DIR}/{portfolio} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()

    # Build a complete skeleton of every 30-min slot so gaps become explicit NaNs
    # rather than silently disappearing from the dataset (from XGBoost model).
    dr       = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })
    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    df['Date']        = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025', format='%B %d %Y')
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['IntervalIdx'] = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )

    # ── Three-level imputation (from XGBoost model, boundary-safe) ───────────
    # transform() ensures bfill/ffill never cross day boundaries.
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        # Step 1: smooth gaps within a day using linear interpolation
        df[col] = df.groupby(['Month', 'Day'])[col].transform(
            lambda x: x.interpolate(method='linear').bfill().ffill()
        )
        # Step 2: same day-of-week + same interval median for missing whole days
        df[col] = df.groupby(['day_of_week', 'IntervalIdx'])[col].transform(
            lambda x: x.fillna(x.median())
        )
        # Step 3: global median as last resort
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)

    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    remaining = df[['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']].isnull().sum().sum()
    print(f"  {portfolio} interval nulls after cleanup: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# SEQUENCE BUILDER  (unchanged from original LSTM)
# ══════════════════════════════════════════════════════════════════════════════
def create_sequences(data, time_steps, num_targets):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i : i + time_steps, :])
        y.append(data[i + time_steps, :num_targets])
    return np.array(X), np.array(y)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
def process_portfolio(portfolio):
    print(f"\n{'='*50}\nProcessing Portfolio {portfolio}\n{'='*50}")

    # ── STAGE 1: Load & prep daily data ──────────────────────────────────────
    d = load_daily(portfolio)

    df_model = d[ALL_COLS].copy()
    scaler   = MinMaxScaler(feature_range=(0, 1))
    scaled   = scaler.fit_transform(df_model)

    time_steps     = 14
    total_features = len(ALL_COLS)

    X_full, y_full = create_sequences(scaled, time_steps, NUM_TARGETS)
    print(f"  Training sequences: {X_full.shape}")

    # ── STAGE 2: Build & train LSTM ───────────────────────────────────────────
    # Architecture is unchanged from original LSTM.
    # The richer feature set does the heavy lifting; no need to change depth.
    K.clear_session()
    model = Sequential([
        Input(shape=(time_steps, total_features)),
        LSTM(64, activation='tanh', return_sequences=True),
        Dropout(0.2),
        LSTM(32, activation='tanh', return_sequences=False),
        Dropout(0.2),
        Dense(NUM_TARGETS, activation='sigmoid'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001, clipvalue=1.0),
        loss=asymmetric_loss
    )
    print(f"  Training LSTM (40 epochs) with asymmetric loss...")
    model.fit(X_full, y_full, epochs=40, batch_size=32, verbose=0)

    # ── STAGE 3: Build August 2026 daily grid ────────────────────────────────
    aug_days = pd.date_range('2026-08-01', '2026-08-31', freq='D')
    aug = pd.DataFrame({'Date': aug_days})

    # Calendar & cyclical features — identical construction to load_daily()
    aug['day_of_week']  = aug['Date'].dt.dayofweek
    aug['month']        = aug['Date'].dt.month
    aug['day_of_month'] = aug['Date'].dt.day
    aug['week_of_month']= (aug['day_of_month'] - 1) // 7 + 1
    aug['is_weekend']   = (aug['day_of_week'] >= 5).astype(int)
    aug['dow_sin']      = np.sin(2 * np.pi * aug['day_of_week'] / 7)
    aug['dow_cos']      = np.cos(2 * np.pi * aug['day_of_week'] / 7)
    aug['month_sin']    = np.sin(2 * np.pi * aug['month'] / 12)
    aug['month_cos']    = np.cos(2 * np.pi * aug['month'] / 12)

    us_holidays      = holidays.US(years=[2026])
    aug['is_holiday'] = aug['Date'].isin(us_holidays).astype(int)

    # 364-day lags: pull exact August 2025 actuals — these are known values,
    # not guesses. This is the key advantage of the lag design.
    aug_2025           = d[(d['Date'].dt.month == 8) & (d['Date'].dt.year == 2025)].reset_index(drop=True)
    aug['cv_lag_364']  = aug_2025['Call Volume'].values
    aug['cct_lag_364'] = aug_2025['CCT'].values
    aug['abd_lag_364'] = aug_2025['Abandon Rate'].values

    # August baseline anchors — same scalars computed from full historical data
    aug_all            = d[d['month'] == 8]
    aug['aug_cv_mean'] = aug_all['Call Volume'].mean()
    aug['aug_cct_mean']= aug_all['CCT'].mean()
    aug['aug_abd_mean']= aug_all['Abandon Rate'].mean()

    aug_dow_means          = aug_all.groupby('day_of_week')[['Call Volume', 'CCT', 'Abandon Rate']].mean()
    aug_dow_means.columns  = ['aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean']
    aug                    = aug.merge(aug_dow_means, on='day_of_week', how='left')

    # Placeholders for targets and workload — filled during the recursive loop
    for t in TARGETS:
        aug[t] = 0.0
    aug['Workload'] = 0.0

    # ── STAGE 4: Recursive daily forecast ────────────────────────────────────
    # For each August day: predict → write back targets + workload → roll window.
    # Workload is updated dynamically (from original LSTM) so tomorrow's prediction
    # sees an accurate staffing-load signal, not a stale zero.
    aug_model      = aug[ALL_COLS].copy()
    current_window = np.expand_dims(scaled[-time_steps:].copy(), axis=0)

    for i in range(len(aug)):
        pred_scaled             = model.predict(current_window, verbose=0)
        dummy_row               = np.zeros((1, total_features))
        dummy_row[0, :NUM_TARGETS] = pred_scaled[0]
        real_pred               = scaler.inverse_transform(dummy_row)[0, :NUM_TARGETS]
        real_pred               = np.maximum(0, real_pred)   # enforce non-negative

        # Write predictions into both aug (for readability) and aug_model (for scaling)
        for j, t in enumerate(TARGETS):
            aug.loc[aug.index[i], t]       = real_pred[j]
            aug_model.loc[aug_model.index[i], t] = real_pred[j]

        # Workload = CV × CCT — must be updated before rolling the window forward
        wl = real_pred[0] * real_pred[1]
        aug.loc[aug.index[i], 'Workload']       = wl
        aug_model.loc[aug_model.index[i], 'Workload'] = wl

        new_row_scaled = scaler.transform(aug_model.iloc[[i]])
        current_window = np.append(current_window[:, 1:, :], [new_row_scaled], axis=1)

    print("  August daily predictions:")
    print(aug[['Date', 'Call Volume', 'CCT', 'Service Level', 'Abandon Rate']].to_string())

    # 15% upward buffer on call volume (from XGBoost model).
    # Asymmetric loss already nudges the model upward during training;
    # this explicit buffer adds a post-hoc staffing safety margin.
    aug['Call Volume'] = aug['Call Volume'] * 1.15

    # ── STAGE 5: Intraday profile disaggregation ─────────────────────────────
    iv = load_interval(portfolio)

    # Merge daily total so we can compute per-interval share
    daily_cv = iv.groupby('Date')['Call Volume'].sum().rename('daily_cv').reset_index()
    iv       = iv.merge(daily_cv, on='Date', how='left')

    # Weight June 2× — most recent month, most representative of near-future patterns
    iv_weighted = pd.concat([iv, iv[iv['Date'].dt.month == 6]], ignore_index=True)

    # Normalised call-volume share profile (from XGBoost model).
    # Normalisation guarantees interval shares sum exactly to 1 per DOW,
    # so sum(interval predictions) == daily prediction for every day.
    iv_weighted['cv_share'] = iv_weighted['Call Volume'] / iv_weighted['daily_cv'].clip(lower=1)
    profile_cv  = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
    dow_sums    = profile_cv.groupby(level='day_of_week').transform('sum')
    profile_cv  = profile_cv / dow_sums.clip(lower=1e-9)

    # CCT and Abandon Rate are absolute interval means, not shares
    profile_cct = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean()
    profile_abd = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean()

    # ── Profile validation on June (MAPE check) ──────────────────────────────
    iv_june = iv[iv['Date'].dt.month == 6].copy()
    iv_june = iv_june.merge(profile_cv.rename('cv_share_pred').reset_index(),
                            on=['day_of_week', 'IntervalIdx'], how='left')
    iv_june = iv_june.merge(daily_cv.rename(columns={'daily_cv': 'actual_daily'}), on='Date', how='left')
    iv_june['cv_pred'] = iv_june['cv_share_pred'] * iv_june['actual_daily']
    mask = iv_june['Call Volume'] > 0
    if mask.sum() > 0:
        mape = mean_absolute_percentage_error(
            iv_june.loc[mask, 'Call Volume'], iv_june.loc[mask, 'cv_pred']
        ) * 100
        print(f"  CV profile MAPE on June: {mape:.2f}%")

    # ── STAGE 6: Generate August interval rows ────────────────────────────────
    rows = []
    for _, day_row in aug.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])

        for iod in range(48):
            h  = iod // 2
            mn = (iod % 2) * 30

            cv_share = profile_cv.get((dow, iod), 1 / 48)
            cv_pred  = max(0, cv_share * day_row['Call Volume'])
            cct_pred = max(0, profile_cct.get((dow, iod), day_row['CCT']))
            abd_pred = float(np.clip(profile_abd.get((dow, iod), 0.01), 0, 1))

            rows.append({
                'Portfolio':       portfolio,
                'Month':           'August',
                'Day':             dom,
                'Interval':        f"{h}:{mn:02d}",
                f'Calls_Offered_{portfolio}':   round(cv_pred, 2),
                f'Abandoned_Calls_{portfolio}': round(cv_pred * abd_pred, 2),
                f'Abandoned_Rate_{portfolio}':  round(abd_pred, 6),
                f'CCT_{portfolio}':             round(cct_pred, 2),
            })

    result = pd.DataFrame(rows)
    print(f"  {portfolio}: {len(rows)} rows generated")
    return result


# ══════════════════════════════════════════════════════════════════════════════
# EXECUTE
# ══════════════════════════════════════════════════════════════════════════════
all_forecasts = []
for p in PORTFOLIOS:
    all_forecasts.append(process_portfolio(p))

# Combine into single wide output (same structure as XGBoost output)
print("\nCombining portfolios...")
result = all_forecasts[0][['Month', 'Day', 'Interval',
    'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A']]

for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(
        all_forecasts[i + 1][[
            'Month', 'Day', 'Interval',
            f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
            f'Abandoned_Rate_{p}', f'CCT_{p}'
        ]],
        on=['Month', 'Day', 'Interval'], how='left'
    )

col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [f'Calls_Offered_{p}', f'Abandoned_Calls_{p}', f'Abandoned_Rate_{p}', f'CCT_{p}']
result = result[col_order]

# Final business-logic clipping
for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved to {OUTPUT_PATH}")
print(f"   Shape:     {result.shape}  |  Expected: (1488, 19)")
print(f"   Negatives: {(result.select_dtypes('number') < 0).any().any()}")
print(f"   Nulls:     {result.isnull().any().any()}")
print("\n=== Daily CV totals by portfolio ===")
for p in PORTFOLIOS:
    col   = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")
print("\n=== First 5 rows ===")
print(result.head(5).to_string())

2026-03-29 16:00:48.459152: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



Processing Portfolio A
  A daily nulls after cleanup: 0
  Training sequences: (717, 14, 24)
  Training LSTM (40 epochs) with asymmetric loss...
  August daily predictions:
         Date  Call Volume         CCT  Service Level  Abandon Rate
0  2026-08-01  6273.798604  321.517661       0.955681      0.014129
1  2026-08-02  6103.939419  325.314435       0.960221      0.012533
2  2026-08-03  4526.140149  330.163962       0.962658      0.012657
3  2026-08-04  2678.662017  329.086681       0.962932      0.011682
4  2026-08-05  6098.921775  327.293041       0.959554      0.011813
5  2026-08-06  5115.349093  328.811825       0.958827      0.012524
6  2026-08-07  4637.611389  328.530710       0.958689      0.013107
7  2026-08-08  5409.247742  324.822485       0.955569      0.013216
8  2026-08-09  5150.959221  326.516119       0.955945      0.013311
9  2026-08-10  3177.470050  329.093139       0.959163      0.013737
10 2026-08-11  1507.179265  328.939799       0.961543      0.013220
11 2026-08-